# ⭐ Day 88: Semantic Segmentation - Pixel-wise Image Understanding with U-Net
## Day 88 of 369-day Python & AI Learning Path

Welcome to Day 88! Today we dive into **Semantic Segmentation**, a powerful Computer Vision technique that enables pixel-level understanding of images. We will master the legendary **U-Net architecture** and build pixel-perfect vision systems! 🖼️🧠

## 📋 Table of Contents

1. [Introduction to Semantic Segmentation vs Object Detection](#1)
2. [Understanding U-Net Architecture](#2)
3. [Loading and Exploring Segmentation Datasets](#3)
4. [Data Preprocessing & Augmentation for Segmentation](#4)
5. [Building U-Net from Scratch](#5)
6. [Training the U-Net Model](#6)
7. [Evaluating Segmentation Performance](#7)
8. [Visualizing Predictions](#8)
9. [Transfer Learning with Pre-trained Segmentation Models](#9)
10. [Real-world Applications & Deployment Ideas](#10)
11. [🛠️ Hands-On Exercises](#11)
12. [✅ Solutions](#12)
13. [Summary & Day 89 Teaser](#13)

## 1. Introduction to Semantic Segmentation vs Object Detection 🖼️ <a id='1'></a>

**Semantic Segmentation** assigns a class label to **every single pixel** in an image, producing a dense pixel-wise classification map. This is fundamentally different from Object Detection, which only draws bounding boxes around objects.

| Task | Output Granularity | Use Case |
|------|------------------|----------|
| **Image Classification** | Single label per image | What is in this image? |
| **Object Detection** | Bounding boxes + labels | Where are the objects? |
| **Semantic Segmentation** | Pixel-level class labels | What is every pixel? |
| **Instance Segmentation** | Pixel masks per object instance | Which pixels belong to which object? |

### Why Segmentation Matters:
- **Medical Imaging**: Precise tumor boundary delineation for diagnosis and surgery planning
- **Autonomous Driving**: Lane detection, drivable area identification, pedestrian boundaries
- **Satellite Imagery**: Land use classification, building footprint extraction, deforestation monitoring
- **Agriculture**: Crop health analysis, weed detection at pixel precision
- **Fashion & Retail**: Virtual try-on, body measurement, fabric segmentation

### Key Challenge:
Unlike classification, segmentation requires **spatial precision** — the model must understand both *what* objects are and *exactly where* their boundaries lie, pixel by pixel.

## 2. Understanding U-Net Architecture 🧠 <a id='2'></a>

**U-Net**, introduced by Ronneberger et al. in 2015 for biomedical image segmentation, has become the gold standard architecture for semantic segmentation. Its name comes from its distinctive U-shaped structure.

### Architecture Components:

#### 🔽 Encoder (Contracting Path / Downsampling)
- **Purpose**: Capture context and semantic information
- **Structure**: Repeated application of two 3×3 convolutions + ReLU + 2×2 max pooling
- **Effect**: Doubles feature channels, halves spatial dimensions at each step
- **Output**: Deep, low-resolution feature maps rich in semantic meaning

#### 🔼 Decoder (Expansive Path / Upsampling)
- **Purpose**: Enable precise localization through progressive upsampling
- **Structure**: 2×2 transpose convolutions (upsampling) + concatenation with skip connections + two 3×3 convolutions
- **Effect**: Halves feature channels, doubles spatial dimensions at each step
- **Output**: High-resolution segmentation map matching input dimensions

#### 🔗 Skip Connections
- **Purpose**: Bridge the gap between semantic information (encoder) and spatial precision (decoder)
- **Mechanism**: Concatenate encoder feature maps with decoder feature maps at corresponding resolutions
- **Benefit**: Preserves fine-grained spatial details that would otherwise be lost during downsampling

### Why U-Net Works So Well:
1. **Symmetrical design** ensures feature map dimensions align perfectly for skip connections
2. **Skip connections** recover spatial precision lost in the bottleneck
3. **End-to-end trainable** with relatively few parameters for its performance
4. **Works with small datasets** thanks to heavy data augmentation and skip connections

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import cv2
import os
import warnings
warnings.filterwarnings('ignore')

print("✅ TensorFlow version:", tf.__version__)
print("✅ GPU Available:", tf.config.list_physical_devices('GPU'))
print("✅ All libraries imported successfully!")

In [ ]:
# Visualize U-Net Architecture Concept
def visualize_unet_architecture():
    """Create a visual diagram of U-Net architecture"""
    fig, ax = plt.subplots(figsize=(14, 10))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    ax.axis('off')
    
    # Title
    ax.text(5, 9.5, '🏗️ U-Net Architecture Overview', fontsize=18, fontweight='bold', 
            ha='center', va='center')
    
    # Encoder blocks (left side, going down)
    encoder_colors = ['#FF6B6B', '#FF8E53', '#FE6B8B', '#FF8E53']
    encoder_sizes = [(2, 1.5), (1.8, 1.3), (1.6, 1.1), (1.4, 0.9)]
    encoder_positions = [(1.5, 7.5), (1.5, 6.0), (1.5, 4.7), (1.5, 3.6)]
    encoder_labels = ['Conv+ReLU\nConv+ReLU\nMaxPool', 'Conv+ReLU\nConv+ReLU\nMaxPool', 
                      'Conv+ReLU\nConv+ReLU\nMaxPool', 'Conv+ReLU\nConv+ReLU\nMaxPool']
    
    for (x, y), (w, h), color, label in zip(encoder_positions, encoder_sizes, encoder_colors, encoder_labels):
        rect = plt.Rectangle((x-w/2, y-h/2), w, h, facecolor=color, edgecolor='black', linewidth=2, alpha=0.8)
        ax.add_patch(rect)
        ax.text(x, y, label, ha='center', va='center', fontsize=8, fontweight='bold', color='white')
    
    # Bottleneck
    bottleneck = plt.Rectangle((1.5-0.6, 2.5-0.5), 1.2, 1, facecolor='#9C27B0', 
                                edgecolor='black', linewidth=2, alpha=0.9)
    ax.add_patch(bottleneck)
    ax.text(1.5, 2.5, 'Bottleneck\nConv+ReLU', ha='center', va='center', 
            fontsize=9, fontweight='bold', color='white')
    
    # Decoder blocks (right side, going up)
    decoder_colors = ['#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7']
    decoder_sizes = [(1.4, 0.9), (1.6, 1.1), (1.8, 1.3), (2, 1.5)]
    decoder_positions = [(8.5, 3.6), (8.5, 4.7), (8.5, 6.0), (8.5, 7.5)]
    decoder_labels = ['UpConv\nConcat\nConv+ReLU', 'UpConv\nConcat\nConv+ReLU',
                      'UpConv\nConcat\nConv+ReLU', 'UpConv\nConcat\nConv+ReLU']
    
    for (x, y), (w, h), color, label in zip(decoder_positions, decoder_sizes, decoder_colors, decoder_labels):
        rect = plt.Rectangle((x-w/2, y-h/2), w, h, facecolor=color, edgecolor='black', linewidth=2, alpha=0.8)
        ax.add_patch(rect)
        ax.text(x, y, label, ha='center', va='center', fontsize=8, fontweight='bold')
    
    # Skip connections (gray lines)
    skip_y = [7.5, 6.0, 4.7, 3.6]
    for y in skip_y:
        ax.annotate('', xy=(8.5-0.7, y), xytext=(1.5+0.7, y),
                   arrowprops=dict(arrowstyle='->', color='gray', lw=1.5, linestyle='--'))
        ax.text(5, y+0.15, 'Skip Connection', ha='center', fontsize=7, color='gray', style='italic')
    
    # Down arrows (encoder)
    for i in range(3):
        y_start = encoder_positions[i][1] - encoder_sizes[i][1]/2
        y_end = encoder_positions[i+1][1] + encoder_sizes[i+1][1]/2
        ax.annotate('', xy=(1.5, y_end), xytext=(1.5, y_start),
                   arrowprops=dict(arrowstyle='->', color='black', lw=2))
    
    # Up arrows (decoder)
    for i in range(3):
        y_start = decoder_positions[i][1] + decoder_sizes[i][1]/2
        y_end = decoder_positions[i+1][1] - decoder_sizes[i+1][1]/2
        ax.annotate('', xy=(8.5, y_end), xytext=(8.5, y_start),
                   arrowprops=dict(arrowstyle='->', color='black', lw=2))
    
    # Center arrow (bottleneck to decoder)
    ax.annotate('', xy=(8.5, 3.6-0.45), xytext=(1.5, 2.5-0.5),
               arrowprops=dict(arrowstyle='->', color='black', lw=2))
    
    # Input and Output
    ax.text(1.5, 8.8, '📥 Input\nImage', ha='center', fontsize=10, fontweight='bold', color='navy')
    ax.text(8.5, 8.8, '📤 Output\nSegmentation\nMask', ha='center', fontsize=10, fontweight='bold', color='darkgreen')
    
    # Labels
    ax.text(1.5, 1.5, '🔽 Encoder\n(Context)', ha='center', fontsize=11, fontweight='bold', color='#D32F2F')
    ax.text(8.5, 1.5, '🔼 Decoder\n(Localize)', ha='center', fontsize=11, fontweight='bold', color='#1976D2')
    
    plt.tight_layout()
    plt.show()

visualize_unet_architecture()

print("\n💡 U-Net Key Insight:")
print("   The skip connections copy high-resolution features from the encoder")
print("   directly to the decoder, preserving spatial precision for accurate boundaries!")

## 3. Loading and Exploring Segmentation Datasets 🗂️ <a id='3'></a>

We'll use the **Oxford-IIIT Pet Dataset**, a popular benchmark for semantic segmentation containing images of cats and dogs with pixel-level foreground/background/border annotations.

In [ ]:
# Download and prepare Oxford-IIIT Pet Dataset (simulated for demonstration)
# In practice: tfds.load('oxford_iiit_pet', split=['train', 'test'])

def create_synthetic_segmentation_dataset(num_samples=200, img_size=128):
    """Create synthetic dataset mimicking segmentation task"""
    np.random.seed(88)
    
    images = []
    masks = []
    
    for i in range(num_samples):
        # Create base image
        img = np.ones((img_size, img_size, 3), dtype=np.float32)
        
        # Random background color
        bg_color = np.random.uniform(0.3, 0.7, 3)
        img *= bg_color
        
        # Create random shape (simulating pet)
        center = (np.random.randint(30, img_size-30), np.random.randint(30, img_size-30))
        axes_len = (np.random.randint(15, 35), np.random.randint(15, 35))
        angle = np.random.randint(0, 180)
        
        # Color for shape
        shape_color = np.random.uniform(0.1, 0.4, 3)
        
        # Draw ellipse on image
        cv2.ellipse(img, center, axes_len, angle, 0, 360, shape_color, -1)
        
        # Create mask: 0=background, 1=foreground, 2=border
        mask = np.zeros((img_size, img_size), dtype=np.uint8)
        cv2.ellipse(mask, center, axes_len, angle, 0, 360, 1, -1)
        
        # Add border (dilation - erosion)
        kernel = np.ones((3, 3), np.uint8)
        dilated = cv2.dilate(mask, kernel, iterations=1)
        eroded = cv2.erode(mask, kernel, iterations=1)
        border = dilated - eroded
        mask[border > 0] = 2
        
        # Add noise
        img += np.random.normal(0, 0.02, img.shape)
        img = np.clip(img, 0, 1)
        
        images.append(img)
        masks.append(mask)
    
    return np.array(images), np.array(masks)

# Generate dataset
print("🔄 Generating synthetic segmentation dataset...")
images, masks = create_synthetic_segmentation_dataset(num_samples=300, img_size=128)

# Split into train/val/test
n_train = 200
n_val = 50
train_images, val_images, test_images = images[:n_train], images[n_train:n_train+n_val], images[n_train+n_val:]
train_masks, val_masks, test_masks = masks[:n_train], masks[n_train:n_train+n_val], masks[n_train+n_val:]

print(f"✅ Dataset created!")
print(f"   📊 Training samples:   {len(train_images)}")
print(f"   📊 Validation samples: {len(val_images)}")
print(f"   📊 Test samples:      {len(test_images)}")
print(f"   🖼️  Image shape:       {train_images[0].shape}")
print(f"   🎭 Mask shape:        {train_masks[0].shape}")
print(f"   🏷️  Classes:           Background(0), Foreground(1), Border(2)")

In [ ]:
# Visualize sample images with their masks
def visualize_dataset_samples(images, masks, num_samples=6):
    """Display original images alongside their segmentation masks"""
    
    fig, axes = plt.subplots(2, num_samples, figsize=(18, 6))
    
    indices = np.random.choice(len(images), num_samples, replace=False)
    
    for idx, sample_idx in enumerate(indices):
        # Original image
        axes[0, idx].imshow(images[sample_idx])
        axes[0, idx].set_title(f'Sample {sample_idx}', fontsize=10, fontweight='bold')
        axes[0, idx].axis('off')
        
        # Mask with color coding
        mask_colored = np.zeros((*masks[sample_idx].shape, 3))
        mask_colored[masks[sample_idx] == 0] = [0.2, 0.2, 0.8]   # Background - blue
        mask_colored[masks[sample_idx] == 1] = [0.2, 0.8, 0.2]   # Foreground - green
        mask_colored[masks[sample_idx] == 2] = [0.9, 0.9, 0.2]   # Border - yellow
        
        axes[1, idx].imshow(mask_colored)
        axes[1, idx].set_title('Ground Truth Mask', fontsize=10, fontweight='bold')
        axes[1, idx].axis('off')
    
    axes[0, 0].set_ylabel('Original\nImage', fontsize=12, fontweight='bold', rotation=0, labelpad=40, va='center')
    axes[1, 0].set_ylabel('Segmentation\nMask', fontsize=12, fontweight='bold', rotation=0, labelpad=40, va='center')
    
    plt.suptitle('🗂️ Dataset Samples: Images & Ground Truth Masks', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_dataset_samples(train_images, train_masks, num_samples=6)

# Class distribution analysis
unique, counts = np.unique(train_masks, return_counts=True)
class_names = ['Background', 'Foreground', 'Border']
class_colors = ['#4169E1', '#32CD32', '#FFD700']

fig, ax = plt.subplots(1, 1, figsize=(8, 5))
bars = ax.bar(range(len(unique)), counts, color=class_colors, edgecolor='black', linewidth=1.5)
ax.set_xticks(range(len(unique)))
ax.set_xticklabels(class_names, fontsize=11)
ax.set_ylabel('Pixel Count', fontsize=12)
ax.set_title('📊 Class Distribution in Training Masks', fontsize=14, fontweight='bold')

# Add percentage labels
total_pixels = counts.sum()
for bar, count in zip(bars, counts):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{count:,}\n({count/total_pixels:.1%})',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n⚠️  Class imbalance detected! Background dominates. We'll use weighted loss.")

## 4. Data Preprocessing & Augmentation for Segmentation 🔄 <a id='4'></a>

Segmentation requires **spatially consistent augmentations** — image and mask must be transformed identically to maintain pixel alignment.

In [ ]:
# Data preprocessing and augmentation pipeline
class SegmentationDataGenerator:
    """Custom data generator with synchronized image-mask augmentation"""
    
    def __init__(self, images, masks, batch_size=16, augment=True):
        self.images = images
        self.masks = masks
        self.batch_size = batch_size
        self.augment = augment
        self.num_samples = len(images)
    
    def __len__(self):
        return self.num_samples // self.batch_size
    
    def _augment_pair(self, image, mask):
        """Apply identical augmentation to image and mask"""
        if not self.augment:
            return image, mask
        
        # Random horizontal flip (synchronized)
        if np.random.random() > 0.5:
            image = np.fliplr(image)
            mask = np.fliplr(mask)
        
        # Random rotation (synchronized)
        if np.random.random() > 0.7:
            angle = np.random.uniform(-15, 15)
            h, w = image.shape[:2]
            center = (w // 2, h // 2)
            M = cv2.getRotationMatrix2D(center, angle, 1.0)
            image = cv2.warpAffine(image, M, (w, h), borderMode=cv2.BORDER_REFLECT)
            mask = cv2.warpAffine(mask.astype(np.float32), M, (w, h), 
                                  borderMode=cv2.BORDER_REFLECT, flags=cv2.INTER_NEAREST)
            mask = mask.astype(np.uint8)
        
        # Random brightness (image only)
        if np.random.random() > 0.5:
            factor = np.random.uniform(0.8, 1.2)
            image = np.clip(image * factor, 0, 1)
        
        # Random zoom (synchronized)
        if np.random.random() > 0.8:
            scale = np.random.uniform(0.9, 1.1)
            h, w = image.shape[:2]
            new_h, new_w = int(h * scale), int(w * scale)
            
            image = cv2.resize(image, (new_w, new_h))
            mask = cv2.resize(mask, (new_w, new_h), interpolation=cv2.INTER_NEAREST)
            
            # Crop or pad to original size
            if scale > 1:  # Crop center
                start_y = (new_h - h) // 2
                start_x = (new_w - w) // 2
                image = image[start_y:start_y+h, start_x:start_x+w]
                mask = mask[start_y:start_y+h, start_x:start_x+w]
            else:  # Pad
                pad_y = (h - new_h) // 2
                pad_x = (w - new_w) // 2
                image = cv2.copyMakeBorder(image, pad_y, h-new_h-pad_y, pad_x, w-new_w-pad_x,
                                          cv2.BORDER_REFLECT)
                mask = cv2.copyMakeBorder(mask, pad_y, h-new_h-pad_y, pad_x, w-new_w-pad_x,
                                         cv2.BORDER_REFLECT)
        
        return image, mask
    
    def __call__(self):
        """Generator for tf.data"""
        while True:
            indices = np.random.permutation(self.num_samples)
            for i in range(0, self.num_samples, self.batch_size):
                batch_indices = indices[i:i+self.batch_size]
                
                batch_images = []
                batch_masks = []
                
                for idx in batch_indices:
                    img, msk = self._augment_pair(self.images[idx].copy(), self.masks[idx].copy())
                    batch_images.append(img)
                    batch_masks.append(msk)
                
                yield np.array(batch_images), np.array(batch_masks)

# Create generators
train_gen = SegmentationDataGenerator(train_images, train_masks, batch_size=16, augment=True)
val_gen = SegmentationDataGenerator(val_images, val_masks, batch_size=16, augment=False)

# Create tf.data datasets
train_dataset = tf.data.Dataset.from_generator(
    train_gen,
    output_signature=(
        tf.TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32),
        tf.TensorSpec(shape=(None, 128, 128), dtype=tf.uint8)
    )
).prefetch(tf.data.AUTOTUNE)

val_dataset = tf.data.Dataset.from_generator(
    val_gen,
    output_signature=(
        tf.TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32),
        tf.TensorSpec(shape=(None, 128, 128), dtype=tf.uint8)
    )
).prefetch(tf.data.AUTOTUNE)

# Visualize augmented samples
fig, axes = plt.subplots(3, 4, figsize=(14, 10))

sample_gen = SegmentationDataGenerator(train_images[:4], train_masks[:4], batch_size=4, augment=True)
batch_img, batch_msk = next(sample_gen())

for i in range(4):
    # Original
    axes[0, i].imshow(train_images[i])
    axes[0, i].set_title(f'Original {i+1}', fontsize=10, fontweight='bold')
    axes[0, i].axis('off')
    
    # Augmented image
    axes[1, i].imshow(batch_img[i])
    axes[1, i].set_title(f'Augmented {i+1}', fontsize=10, fontweight='bold')
    axes[1, i].axis('off')
    
    # Augmented mask
    mask_colored = np.zeros((128, 128, 3))
    mask_colored[batch_msk[i] == 0] = [0.2, 0.2, 0.8]
    mask_colored[batch_msk[i] == 1] = [0.2, 0.8, 0.2]
    mask_colored[batch_msk[i] == 2] = [0.9, 0.9, 0.2]
    axes[2, i].imshow(mask_colored)
    axes[2, i].set_title(f'Aug Mask {i+1}', fontsize=10, fontweight='bold')
    axes[2, i].axis('off')

axes[0, 0].set_ylabel('Original', fontsize=11, fontweight='bold', rotation=0, labelpad=30, va='center')
axes[1, 0].set_ylabel('Augmented\nImage', fontsize=11, fontweight='bold', rotation=0, labelpad=30, va='center')
axes[2, 0].set_ylabel('Augmented\nMask', fontsize=11, fontweight='bold', rotation=0, labelpad=30, va='center')

plt.suptitle('🔄 Synchronized Image-Mask Augmentation', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print("✅ Augmentation pipeline ready!")
print("   🔄 Flips, rotations, brightness, and zoom applied identically to images & masks")

## 5. Building U-Net from Scratch 🏗️ <a id='5'></a>

Now we construct the complete U-Net architecture using Keras Functional API for maximum flexibility.

In [ ]:
def build_unet(input_shape=(128, 128, 3), num_classes=3, dropout_rate=0.1):
    """
    Build U-Net architecture from scratch
    
    Args:
        input_shape: Input image dimensions
        num_classes: Number of segmentation classes
        dropout_rate: Dropout for regularization
    """
    inputs = layers.Input(shape=input_shape)
    
    # Helper functions
    def conv_block(x, num_filters, dropout=False):
        """Two convolutions with BatchNorm and ReLU"""
        x = layers.Conv2D(num_filters, 3, padding='same', kernel_initializer='he_normal')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        
        x = layers.Conv2D(num_filters, 3, padding='same', kernel_initializer='he_normal')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        
        if dropout:
            x = layers.Dropout(dropout_rate)(x)
        return x
    
    def encoder_block(x, num_filters, dropout=False):
        """Conv block + MaxPooling for downsampling"""
        x = conv_block(x, num_filters, dropout)
        p = layers.MaxPooling2D((2, 2))(x)
        return x, p  # x for skip connection, p for next block
    
    def decoder_block(x, skip_features, num_filters, dropout=False):
        """Upsampling + Concatenation + Conv block"""
        x = layers.Conv2DTranspose(num_filters, (2, 2), strides=2, padding='same')(x)
        x = layers.Concatenate()([x, skip_features])
        x = conv_block(x, num_filters, dropout)
        return x
    
    # ========== ENCODER ==========
    # Block 1: 128x128 -> 64x64
    s1, p1 = encoder_block(inputs, 64)
    
    # Block 2: 64x64 -> 32x32
    s2, p2 = encoder_block(p1, 128)
    
    # Block 3: 32x32 -> 16x16
    s3, p3 = encoder_block(p2, 256)
    
    # Block 4: 16x16 -> 8x8
    s4, p4 = encoder_block(p3, 512, dropout=True)
    
    # ========== BOTTLENECK ==========
    b = conv_block(p4, 1024, dropout=True)
    
    # ========== DECODER ==========
    # Block 4: 8x8 -> 16x16
    d4 = decoder_block(b, s4, 512, dropout=True)
    
    # Block 3: 16x16 -> 32x32
    d3 = decoder_block(d4, s3, 256)
    
    # Block 2: 32x32 -> 64x64
    d2 = decoder_block(d3, s2, 128)
    
    # Block 1: 64x64 -> 128x128
    d1 = decoder_block(d2, s1, 64)
    
    # ========== OUTPUT ==========
    outputs = layers.Conv2D(num_classes, 1, padding='same', activation='softmax', name='segmentation_output')(d1)
    
    model = keras.Model(inputs, outputs, name='U-Net')
    return model

# Build model
unet_model = build_unet(input_shape=(128, 128, 3), num_classes=3)

# Display architecture summary
print("🏗️ U-Net Architecture Summary:")
print("=" * 60)
unet_model.summary()

# Visualize model structure
keras.utils.plot_model(unet_model, show_shapes=True, show_layer_names=True, 
                       to_file='/tmp/unet_arch.png', dpi=80)
print("\n✅ U-Net model built successfully!")
print(f"   📊 Total parameters: {unet_model.count_params():,}")
print(f"   🎯 Output shape: {unet_model.output_shape}")

In [ ]:
# Visualize feature map dimensions through the network
def visualize_unet_dimensions():
    """Create a table showing tensor shapes through U-Net"""
    
    stages = [
        ("Input Image", "128×128×3", "📥"),
        ("Encoder Block 1", "128×128×64", "🔽"),
        ("Pool 1", "64×64×64", "🔽"),
        ("Encoder Block 2", "64×64×128", "🔽"),
        ("Pool 2", "32×32×128", "🔽"),
        ("Encoder Block 3", "32×32×256", "🔽"),
        ("Pool 3", "16×16×256", "🔽"),
        ("Encoder Block 4", "16×16×512", "🔽"),
        ("Pool 4", "8×8×512", "🔽"),
        ("Bottleneck", "8×8×1024", "🎯"),
        ("UpSample + Skip 4", "16×16×1024 → 16×16×512", "🔼"),
        ("Decoder Block 4", "16×16×512", "🔼"),
        ("UpSample + Skip 3", "32×32×512 → 32×32×256", "🔼"),
        ("Decoder Block 3", "32×32×256", "🔼"),
        ("UpSample + Skip 2", "64×64×256 → 64×64×128", "🔼"),
        ("Decoder Block 2", "64×64×128", "🔼"),
        ("UpSample + Skip 1", "128×128×128 → 128×128×64", "🔼"),
        ("Decoder Block 1", "128×128×64", "🔼"),
        ("Output", "128×128×3", "📤")
    ]
    
    fig, ax = plt.subplots(figsize=(10, 12))
    ax.axis('off')
    
    # Create table
    table_data = [[stage[2], stage[0], stage[1]] for stage in stages]
    table = ax.table(cellText=table_data, colLabels=['', 'Stage', 'Tensor Shape'],
                     cellLoc='left', loc='center', colWidths=[0.1, 0.45, 0.45])
    
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1, 2)
    
    # Color coding
    for i in range(len(stages) + 1):
        for j in range(3):
            cell = table[(i, j)]
            if i == 0:
                cell.set_facecolor('#4A90E2')
                cell.set_text_props(color='white', fontweight='bold')
            elif i <= 9:
                cell.set_facecolor('#FF6B6B')
                cell.set_text_props(color='white')
            elif i == 10:
                cell.set_facecolor('#9C27B0')
                cell.set_text_props(color='white')
            else:
                cell.set_facecolor('#4ECDC4')
                cell.set_text_props(color='black')
    
    ax.set_title('📐 U-Net Tensor Dimensions Flow', fontsize=16, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.show()

visualize_unet_dimensions()

print("\n💡 Notice the symmetry: each encoder block's output shape matches")
print("   the corresponding decoder block's input shape after upsampling!")

## 6. Training the U-Net Model 🚀 <a id='6'></a>

We'll compile the model with appropriate loss functions and metrics for segmentation, then train with our augmented data generator.

In [ ]:
# Custom metrics for segmentation
def iou_score(y_true, y_pred, smooth=1e-6):
    """Intersection over Union metric"""
    y_true_f = tf.reshape(tf.one_hot(tf.cast(y_true, tf.int32), depth=3), [-1, 3])
    y_pred_f = tf.reshape(y_pred, [-1, 3])
    
    intersection = tf.reduce_sum(y_true_f * y_pred_f, axis=0)
    union = tf.reduce_sum(y_true_f, axis=0) + tf.reduce_sum(y_pred_f, axis=0) - intersection
    
    iou = (intersection + smooth) / (union + smooth)
    return tf.reduce_mean(iou)

def dice_coefficient(y_true, y_pred, smooth=1e-6):
    """Dice coefficient (F1 score for segmentation)"""
    y_true_f = tf.reshape(tf.one_hot(tf.cast(y_true, tf.int32), depth=3), [-1, 3])
    y_pred_f = tf.reshape(y_pred, [-1, 3])
    
    intersection = tf.reduce_sum(y_true_f * y_pred_f, axis=0)
    dice = (2. * intersection + smooth) / (tf.reduce_sum(y_true_f, axis=0) + tf.reduce_sum(y_pred_f, axis=0) + smooth)
    return tf.reduce_mean(dice)

# Weighted loss to handle class imbalance
def weighted_categorical_crossentropy(weights):
    """Create weighted categorical crossentropy loss"""
    weights = tf.constant(weights, dtype=tf.float32)
    
    def loss(y_true, y_pred):
        y_true_one_hot = tf.one_hot(tf.cast(y_true, tf.int32), depth=3)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        
        loss = -tf.reduce_sum(y_true_one_hot * tf.math.log(y_pred) * weights, axis=-1)
        return tf.reduce_mean(loss)
    
    return loss

# Calculate class weights (inverse frequency)
unique, counts = np.unique(train_masks, return_counts=True)
total = counts.sum()
class_weights = [total / (len(unique) * c) for c in counts]
class_weights = np.array(class_weights) / max(class_weights)  # Normalize

print("⚖️ Class Weights for Balanced Loss:")
for i, (cls, w) in enumerate(zip(['Background', 'Foreground', 'Border'], class_weights)):
    print(f"   {cls:12s}: {w:.3f}")

# Compile model
unet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss=weighted_categorical_crossentropy(class_weights),
    metrics=[
        'accuracy',
        iou_score,
        dice_coefficient
    ]
)

print("\n✅ Model compiled with:")
print("   🎯 Loss: Weighted Categorical Crossentropy")
print("   📊 Metrics: Accuracy, IoU, Dice Coefficient")
print("   ⚡ Optimizer: Adam (lr=1e-4)")

In [ ]:
# Training with simulated history (for demonstration without long runtime)
# In practice: history = unet_model.fit(train_dataset, validation_data=val_dataset, epochs=50)

def simulate_training_history(epochs=50):
    """Generate realistic training curves for segmentation"""
    np.random.seed(88)
    
    # Loss curves (decreasing)
    train_loss = 2.0 * np.exp(-np.linspace(0, 2.5, epochs)) + 0.15 + np.random.normal(0, 0.03, epochs)
    val_loss = train_loss + 0.08 + np.random.normal(0, 0.02, epochs)
    
    # Accuracy (increasing)
    train_acc = 1 - 0.6 * np.exp(-np.linspace(0, 2.2, epochs)) + np.random.normal(0, 0.01, epochs)
    val_acc = train_acc - 0.03 + np.random.normal(0, 0.008, epochs)
    
    # IoU (increasing)
    train_iou = 1 - 0.7 * np.exp(-np.linspace(0, 2.0, epochs)) + np.random.normal(0, 0.015, epochs)
    val_iou = train_iou - 0.04 + np.random.normal(0, 0.01, epochs)
    
    # Dice (increasing)
    train_dice = 1 - 0.65 * np.exp(-np.linspace(0, 2.0, epochs)) + np.random.normal(0, 0.012, epochs)
    val_dice = train_dice - 0.03 + np.random.normal(0, 0.008, epochs)
    
    return {
        'loss': np.clip(train_loss, 0.1, 2.0),
        'val_loss': np.clip(val_loss, 0.1, 2.0),
        'accuracy': np.clip(train_acc, 0.4, 0.99),
        'val_accuracy': np.clip(val_acc, 0.4, 0.99),
        'iou_score': np.clip(train_iou, 0.3, 0.95),
        'val_iou_score': np.clip(val_iou, 0.3, 0.95),
        'dice_coefficient': np.clip(train_dice, 0.3, 0.97),
        'val_dice_coefficient': np.clip(val_dice, 0.3, 0.97)
    }

history = simulate_training_history(50)

# Plot training curves
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Loss
axes[0, 0].plot(history['loss'], label='Train Loss', color='blue', linewidth=2)
axes[0, 0].plot(history['val_loss'], label='Val Loss', color='orange', linewidth=2, linestyle='--')
axes[0, 0].set_title('📉 Loss Curves', fontsize=13, fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Accuracy
axes[0, 1].plot(history['accuracy'], label='Train Accuracy', color='green', linewidth=2)
axes[0, 1].plot(history['val_accuracy'], label='Val Accuracy', color='red', linewidth=2, linestyle='--')
axes[0, 1].set_title('📈 Pixel Accuracy', fontsize=13, fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_ylim(0, 1)

# IoU
axes[1, 0].plot(history['iou_score'], label='Train IoU', color='purple', linewidth=2)
axes[1, 0].plot(history['val_iou_score'], label='Val IoU', color='brown', linewidth=2, linestyle='--')
axes[1, 0].set_title('🎯 Intersection over Union (IoU)', fontsize=13, fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('IoU Score')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_ylim(0, 1)

# Dice Coefficient
axes[1, 1].plot(history['dice_coefficient'], label='Train Dice', color='teal', linewidth=2)
axes[1, 1].plot(history['val_dice_coefficient'], label='Val Dice', color='coral', linewidth=2, linestyle='--')
axes[1, 1].set_title('🎲 Dice Coefficient', fontsize=13, fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Dice Score')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_ylim(0, 1)

plt.suptitle('🚀 U-Net Training History', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Final metrics summary
print("\n🏆 Final Training Results (Epoch 50):")
print("=" * 50)
metrics_final = {
    'Val Loss': f"{history['val_loss'][-1]:.4f}",
    'Val Accuracy': f"{history['val_accuracy'][-1]:.3f}",
    'Val IoU': f"{history['val_iou_score'][-1]:.3f}",
    'Val Dice': f"{history['val_dice_coefficient'][-1]:.3f}"
}
for metric, value in metrics_final.items():
    print(f"   {metric:15s}: {value}")
print("=" * 50)
print("💡 Note: In practice, run model.fit() for 50-100 epochs with early stopping")

## 7. Evaluating Segmentation Performance 📊 <a id='7'></a>

Segmentation evaluation goes beyond simple accuracy. We use specialized metrics that account for class imbalance and spatial overlap quality.

In [ ]:
# Comprehensive evaluation metrics visualization
def calculate_segmentation_metrics(y_true, y_pred, num_classes=3):
    """Calculate per-class and mean metrics"""
    
    # Flatten
    y_true_flat = y_true.flatten()
    y_pred_flat = y_pred.flatten()
    
    metrics = {}
    
    for cls in range(num_classes):
        # Binary masks for this class
        true_binary = (y_true_flat == cls).astype(np.float32)
        pred_binary = (y_pred_flat == cls).astype(np.float32)
        
        # Intersection and Union
        intersection = np.sum(true_binary * pred_binary)
        union = np.sum(true_binary) + np.sum(pred_binary) - intersection
        
        # IoU
        iou = intersection / (union + 1e-6)
        
        # Dice
        dice = (2 * intersection) / (np.sum(true_binary) + np.sum(pred_binary) + 1e-6)
        
        # Precision & Recall
        tp = intersection
        fp = np.sum(pred_binary) - intersection
        fn = np.sum(true_binary) - intersection
        
        precision = tp / (tp + fp + 1e-6)
        recall = tp / (tp + fn + 1e-6)
        
        metrics[cls] = {
            'IoU': iou,
            'Dice': dice,
            'Precision': precision,
            'Recall': recall,
            'Pixel_Accuracy': np.sum((y_true_flat == cls) & (y_pred_flat == cls)) / np.sum(true_binary + 1e-6)
        }
    
    # Mean metrics
    metrics['mean'] = {
        'IoU': np.mean([metrics[c]['IoU'] for c in range(num_classes)]),
        'Dice': np.mean([metrics[c]['Dice'] for c in range(num_classes)]),
        'Precision': np.mean([metrics[c]['Precision'] for c in range(num_classes)]),
        'Recall': np.mean([metrics[c]['Recall'] for c in range(num_classes)])
    }
    
    return metrics

# Generate predictions on test set (simulated for demonstration)
def simulate_predictions(test_images, test_masks, noise_level=0.15):
    """Simulate model predictions with realistic accuracy"""
    predictions = []
    
    for img, true_mask in zip(test_images, test_masks):
        # Simulate prediction: mostly correct with some noise
        pred = true_mask.copy()
        
        # Add random misclassifications at boundaries
        noise_mask = np.random.random(pred.shape) < noise_level
        pred[noise_mask] = np.random.randint(0, 3, size=np.sum(noise_mask))
        
        # Smooth boundaries slightly
        pred = cv2.GaussianBlur(pred.astype(np.float32), (3, 3), 0)
        pred = np.round(pred).astype(np.uint8)
        pred = np.clip(pred, 0, 2)
        
        predictions.append(pred)
    
    return np.array(predictions)

test_preds = simulate_predictions(test_images, test_masks, noise_level=0.12)

# Calculate metrics
test_metrics = calculate_segmentation_metrics(test_masks, test_preds)

# Visualize metrics
class_names = ['Background', 'Foreground', 'Border']
colors = ['#4169E1', '#32CD32', '#FFD700']
metric_names = ['IoU', 'Dice', 'Precision', 'Recall']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, metric_name in enumerate(metric_names):
    values = [test_metrics[c][metric_name] for c in range(3)]
    values.append(test_metrics['mean'][metric_name])
    
    labels = class_names + ['Mean']
    bar_colors = colors + ['#FF6B6B']
    
    bars = axes[idx].bar(labels, values, color=bar_colors, edgecolor='black', linewidth=1.5)
    axes[idx].set_ylim(0, 1)
    axes[idx].set_title(f'{metric_name} Score', fontsize=13, fontweight='bold')
    axes[idx].set_ylabel('Score')
    axes[idx].grid(True, alpha=0.3, axis='y')
    
    # Add value labels
    for bar, val in zip(bars, values):
        axes[idx].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                      f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.suptitle('📊 Segmentation Evaluation Metrics (Per-Class & Mean)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

# Print detailed table
print("\n📋 Detailed Segmentation Metrics:")
print("=" * 70)
print(f"{'Class':<15} {'IoU':>8} {'Dice':>8} {'Precision':>10} {'Recall':>8}")
print("-" * 70)
for cls in range(3):
    m = test_metrics[cls]
    print(f"{class_names[cls]:<15} {m['IoU']:>8.3f} {m['Dice']:>8.3f} {m['Precision']:>10.3f} {m['Recall']:>8.3f}")
print("-" * 70)
m = test_metrics['mean']
print(f"{'MEAN':<15} {m['IoU']:>8.3f} {m['Dice']:>8.3f} {m['Precision']:>10.3f} {m['Recall']:>8.3f}")
print("=" * 70)

print("\n💡 Metric Interpretations:")
print("   • IoU > 0.70  → Good segmentation quality")
print("   • Dice > 0.80  → Excellent overlap between prediction and ground truth")
print("   • Precision measures false positives (over-segmentation)")
print("   • Recall measures false negatives (under-segmentation)")

## 8. Visualizing Predictions 🎨 <a id='8'></a>

The most intuitive way to evaluate segmentation is visual comparison: Original Image → Ground Truth → Predicted Mask.

In [ ]:
# Rich visualization: Original | Ground Truth | Prediction | Overlay
def visualize_predictions(images, true_masks, pred_masks, num_samples=6):
    """Create comprehensive prediction visualization"""
    
    indices = np.random.choice(len(images), num_samples, replace=False)
    
    fig, axes = plt.subplots(4, num_samples, figsize=(18, 12))
    
    # Color maps
    mask_colors = np.array([
        [0.2, 0.2, 0.8],   # Background - Blue
        [0.2, 0.8, 0.2],   # Foreground - Green
        [0.9, 0.9, 0.2]    # Border - Yellow
    ])
    
    for idx, sample_idx in enumerate(indices):
        img = images[sample_idx]
        true = true_masks[sample_idx]
        pred = pred_masks[sample_idx]
        
        # Row 1: Original Image
        axes[0, idx].imshow(img)
        axes[0, idx].set_title(f'Test Sample {sample_idx}', fontsize=10, fontweight='bold')
        axes[0, idx].axis('off')
        
        # Row 2: Ground Truth Mask
        true_colored = mask_colors[true]
        axes[1, idx].imshow(true_colored)
        axes[1, idx].set_title('Ground Truth', fontsize=10, fontweight='bold')
        axes[1, idx].axis('off')
        
        # Row 3: Predicted Mask
        pred_colored = mask_colors[pred]
        axes[2, idx].imshow(pred_colored)
        
        # Calculate per-sample IoU for title
        sample_iou = calculate_segmentation_metrics(
            true[np.newaxis, ...], pred[np.newaxis, ...]
        )['mean']['IoU']
        axes[2, idx].set_title(f'Prediction\n(mIoU: {sample_iou:.3f})', fontsize=10, fontweight='bold')
        axes[2, idx].axis('off')
        
        # Row 4: Overlay (Prediction on Image)
        overlay = img.copy()
        # Highlight differences: correct=green tint, wrong=red tint
        correct = (true == pred)
        wrong = ~correct
        
        overlay[correct] = overlay[correct] * 0.7 + np.array([0.2, 0.8, 0.2]) * 0.3
        overlay[wrong] = overlay[wrong] * 0.7 + np.array([0.9, 0.2, 0.2]) * 0.3
        
        axes[3, idx].imshow(np.clip(overlay, 0, 1))
        axes[3, idx].set_title('Error Overlay\n(Green=Correct, Red=Wrong)', fontsize=9, fontweight='bold')
        axes[3, idx].axis('off')
    
    # Row labels
    row_labels = ['📷 Original', '🎯 Ground Truth', '🔮 Prediction', '🔍 Error Overlay']
    for i, label in enumerate(row_labels):
        axes[i, 0].set_ylabel(label, fontsize=11, fontweight='bold', rotation=0, 
                             labelpad=50, va='center')
    
    plt.suptitle('🎨 Segmentation Prediction Visualization', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_predictions(test_images, test_masks, test_preds, num_samples=6)

# Pixel-wise error analysis
def visualize_error_heatmap(true_masks, pred_masks):
    """Visualize where errors occur most frequently"""
    
    error_maps = []
    for t, p in zip(true_masks, pred_masks):
        error = (t != p).astype(np.float32)
        error_maps.append(error)
    
    mean_error = np.mean(error_maps, axis=0)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Mean error heatmap
    im1 = axes[0].imshow(mean_error, cmap='hot', vmin=0, vmax=1)
    axes[0].set_title('🔥 Mean Error Heatmap\n(Red = Frequent Errors)', fontsize=12, fontweight='bold')
    axes[0].axis('off')
    plt.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)
    
    # Error by class
    class_errors = []
    for cls in range(3):
        cls_mask = (true_masks == cls)
        cls_errors = np.sum((true_masks != pred_masks) & cls_mask) / (np.sum(cls_mask) + 1e-6)
        class_errors.append(cls_errors)
    
    bars = axes[1].bar(class_names, class_errors, color=colors, edgecolor='black', linewidth=1.5)
    axes[1].set_title('📊 Error Rate by Class', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('Error Rate')
    axes[1].set_ylim(0, max(class_errors) * 1.2)
    
    for bar, val in zip(bars, class_errors):
        axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                    f'{val:.2%}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    
    plt.suptitle('🔍 Segmentation Error Analysis', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualize_error_heatmap(test_masks, test_preds)

print("\n💡 Key Insight: Errors concentrate at object boundaries (Border class),")
print("   which is expected and common in segmentation tasks.")

## 9. Transfer Learning with Pre-trained Segmentation Models 🔄 <a id='9'></a>

For production applications, we can leverage pre-trained segmentation models and fine-tune them on our specific domain.

In [ ]:
# Demonstrate transfer learning approach with pre-trained backbone
def build_unet_with_pretrained_backbone(input_shape=(128, 128, 3), num_classes=3):
    """
    Build U-Net with pre-trained MobileNetV2 as encoder backbone
    This leverages ImageNet features for better initialization
    """
    
    # Pre-trained encoder (backbone)
    backbone = keras.applications.MobileNetV2(
        input_shape=input_shape,
        include_top=False,
        weights='imagenet'
    )
    
    # Extract specific layers for skip connections
    layer_names = [
        'block_1_expand_relu',   # 64x64
        'block_3_expand_relu',   # 32x32
        'block_6_expand_relu',   # 16x16
        'block_13_expand_relu',  # 8x8
    ]
    
    layers_list = [backbone.get_layer(name).output for name in layer_names]
    
    # Create encoder model
    encoder = keras.Model(inputs=backbone.input, outputs=layers_list)
    
    # Build decoder
    inputs = layers.Input(shape=input_shape)
    encoder_outputs = encoder(inputs)
    
    # Decoder with skip connections
    x = encoder_outputs[-1]  # Bottleneck (8x8)
    
    # Decoder block 1: 8x8 -> 16x16
    x = layers.Conv2DTranspose(256, (2, 2), strides=2, padding='same')(x)
    x = layers.Concatenate()([x, encoder_outputs[2]])
    x = layers.Conv2D(256, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(256, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    
    # Decoder block 2: 16x16 -> 32x32
    x = layers.Conv2DTranspose(128, (2, 2), strides=2, padding='same')(x)
    x = layers.Concatenate()([x, encoder_outputs[1]])
    x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    
    # Decoder block 3: 32x32 -> 64x64
    x = layers.Conv2DTranspose(64, (2, 2), strides=2, padding='same')(x)
    x = layers.Concatenate()([x, encoder_outputs[0]])
    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    
    # Final upsampling: 64x64 -> 128x128
    x = layers.Conv2DTranspose(32, (2, 2), strides=2, padding='same')(x)
    x = layers.Conv2D(32, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    
    # Output
    outputs = layers.Conv2D(num_classes, 1, padding='same', activation='softmax')(x)
    
    model = keras.Model(inputs, outputs, name='U-Net_MobileNetV2')
    return model

# Build transfer learning model
tl_model = build_unet_with_pretrained_backbone()

print("🔄 Transfer Learning U-Net Built!")
print(f"   Backbone: MobileNetV2 (ImageNet pre-trained)")
print(f"   Total parameters: {tl_model.count_params():,}")
print(f"   Pre-trained encoder frozen for initial training")

# Freeze encoder initially
for layer in tl_model.layers:
    if 'model' in layer.name:  # The backbone model
        layer.trainable = False

print(f"   Trainable parameters after freezing: {sum([tf.keras.backend.count_params(w) for w in tl_model.trainable_weights]):,}")
print(f"   Non-trainable parameters: {sum([tf.keras.backend.count_params(w) for w in tl_model.non_trainable_weights]):,}")

# Training strategy visualization
fig, ax = plt.subplots(figsize=(12, 6))
ax.axis('off')

strategy_text = """
🎯 Transfer Learning Training Strategy
═══════════════════════════════════════════════════════════════

Phase 1: Feature Extraction (Epochs 1-20)
   • Freeze pre-trained encoder (backbone)
   • Train only decoder (randomly initialized)
   • Learning rate: 1e-3
   • Goal: Adapt decoder to your dataset's spatial patterns

Phase 2: Fine-Tuning (Epochs 21-50)
   • Unfreeze entire network (or top layers of encoder)
   • Use very low learning rate: 1e-5
   • Goal: Fine-tune pre-trained features for your domain
   • Benefit: Prevents catastrophic forgetting of ImageNet features

Phase 3: Full Training (Optional, Epochs 51+)
   • Train all layers with standard schedule
   • Apply heavy data augmentation
   • Goal: Squeeze out final performance gains

💡 Why This Works:
   ImageNet features (edges, textures, shapes) transfer well to
   most vision tasks. The decoder learns to map these features to
   your specific segmentation classes.
"""

ax.text(0.05, 0.5, strategy_text, fontsize=11, family='monospace',
        verticalalignment='center', bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))
ax.set_title('📚 Transfer Learning Best Practices', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Real-world Applications & Deployment Ideas 🌐 <a id='10'></a>

Semantic Segmentation powers critical applications across industries:

### 🏥 Healthcare & Medical Imaging
- **Tumor Segmentation**: Precise boundary delineation for radiation therapy planning
- **Organ Segmentation**: Liver, kidney, heart segmentation for surgical navigation
- **Cell Segmentation**: Microscopy image analysis for drug discovery
- **COVID-19 Detection**: Lung lesion segmentation from CT scans

### 🚗 Autonomous Vehicles
- **Drivable Area Detection**: Road vs. sidewalk vs. obstacle segmentation
- **Lane Detection**: Pixel-precise lane boundary identification
- **Free Space Detection**: Identifying safe areas for vehicle movement
- **Panoptic Segmentation**: Combining semantic and instance segmentation

### 🛰️ Remote Sensing & Satellite Imagery
- **Land Use Classification**: Urban, forest, water, agriculture mapping
- **Building Extraction**: Automated 3D city modeling from aerial imagery
- **Disaster Response**: Flood damage assessment, fire spread monitoring
- **Agriculture**: Crop health monitoring, yield prediction, irrigation planning

### 🏭 Manufacturing & Industry
- **Defect Detection**: Surface crack, scratch, and anomaly segmentation
- **Quality Control**: Component placement verification on PCBs
- **Robotics**: Grasp point detection, object manipulation boundaries

### 🎮 Entertainment & Creative
- **Background Removal**: Real-time portrait segmentation for video calls
- **Virtual Try-On**: Clothing segmentation for AR shopping experiences
- **Photo Editing**: Selective editing based on semantic regions
- **Game Development**: Procedural terrain generation from satellite data

### 💻 Deployment Strategies
| Platform | Framework | Optimization |
|----------|-----------|--------------|
| Mobile (iOS/Android) | TensorFlow Lite, Core ML | INT8 quantization, pruning |
| Edge (Jetson, Coral) | TensorRT, ONNX Runtime | FP16 inference, batching |
| Web Browser | TensorFlow.js, ONNX.js | WASM acceleration, WebGL |
| Cloud API | TensorFlow Serving, TorchServe | Dynamic batching, caching |
| Embedded | OpenVINO, NCNN | Model distillation, layer fusion |

In [ ]:
# Deployment-ready inference pipeline
class SegmentationDeployer:
    """Production-ready segmentation inference wrapper"""
    
    def __init__(self, model, class_names=None, input_size=(128, 128)):
        self.model = model
        self.input_size = input_size
        self.class_names = class_names or ['Background', 'Foreground', 'Border']
        self.class_colors = np.array([
            [0.2, 0.2, 0.8],
            [0.2, 0.8, 0.2],
            [0.9, 0.9, 0.2]
        ])
        print(f"🚀 Segmentation Deployer initialized")
        print(f"   Input size: {input_size}")
        print(f"   Classes: {self.class_names}")
    
    def preprocess(self, image):
        """Preprocess image for inference"""
        if image.max() > 1:
            image = image / 255.0
        
        # Resize
        if image.shape[:2] != self.input_size:
            image = cv2.resize(image, self.input_size)
        
        # Add batch dimension if needed
        if len(image.shape) == 3:
            image = np.expand_dims(image, axis=0)
        
        return image.astype(np.float32)
    
    def predict(self, image):
        """Run inference and return structured results"""
        preprocessed = self.preprocess(image)
        prediction = self.model.predict(preprocessed, verbose=0)
        
        # Get class predictions
        pred_mask = np.argmax(prediction[0], axis=-1)
        confidence = np.max(prediction[0], axis=-1)
        
        # Calculate class distribution
        class_distribution = {}
        total_pixels = pred_mask.size
        for i, name in enumerate(self.class_names):
            count = np.sum(pred_mask == i)
            class_distribution[name] = {
                'pixels': int(count),
                'percentage': float(count / total_pixels)
            }
        
        return {
            'mask': pred_mask,
            'confidence_map': confidence,
            'class_distribution': class_distribution,
            'mean_confidence': float(np.mean(confidence))
        }
    
    def visualize(self, image, result, save_path=None):
        """Create visualization of segmentation result"""
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        
        # Original
        axes[0].imshow(image)
        axes[0].set_title('📷 Input Image', fontsize=12, fontweight='bold')
        axes[0].axis('off')
        
        # Segmentation mask
        mask_colored = self.class_colors[result['mask']]
        axes[1].imshow(mask_colored)
        axes[1].set_title('🎭 Segmentation Mask', fontsize=12, fontweight='bold')
        axes[1].axis('off')
        
        # Overlay
        overlay = image * 0.6 + mask_colored * 0.4
        axes[2].imshow(np.clip(overlay, 0, 1))
        axes[2].set_title(f'🔮 Overlay (conf: {result["mean_confidence"]:.2f})', 
                         fontsize=12, fontweight='bold')
        axes[2].axis('off')
        
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            print(f"💾 Saved visualization to {save_path}")
        
        plt.show()
        
        # Print statistics
        print("\n📊 Segmentation Statistics:")
        for cls, stats in result['class_distribution'].items():
            print(f"   {cls:12s}: {stats['pixels']:>6,} pixels ({stats['percentage']:>6.1%})")

# Test deployment pipeline
deployer = SegmentationDeployer(unet_model, input_size=(128, 128))

# Test on a sample
sample_img = test_images[0]
result = deployer.predict(sample_img)
deployer.visualize(sample_img, result)

print("\n✅ Deployment pipeline ready for production!")
print("   💡 Wrap in FastAPI/Flask for REST API deployment")
print("   💡 Convert to TFLite for mobile/edge deployment")
print("   💡 Use ONNX Runtime for cross-platform inference")

## 🛠️ Hands-On Exercises <a id='11'></a>

Test your segmentation mastery with these practical challenges!

### Exercise 1: 🔍 Multi-Scale Evaluation
Implement a function that evaluates the U-Net model at multiple input resolutions (64×64, 128×128, 256×256). Compare how resolution affects IoU scores and inference time. Plot a trade-off curve showing accuracy vs. speed.

### Exercise 2: 🎨 Custom Color Map Generator
Build a function that takes a predicted segmentation mask and generates a custom visualization where each class is rendered with a unique color, pattern (stripes, dots, checkerboard), and transparency level. Add a legend showing class names, colors, and pixel counts.

### Exercise 3: 🔄 Post-Processing Pipeline
Create a post-processing pipeline that applies morphological operations (opening, closing, hole filling) to predicted masks to remove noise and smooth boundaries. Compare IoU scores before and after post-processing using side-by-side visualizations.

### Exercise 4: 📊 Class-Wise Performance Dashboard
Build a comprehensive dashboard that displays per-class metrics (IoU, Dice, Precision, Recall) as interactive radar charts and bar plots. Include a confusion matrix for pixel-level classification and highlight the worst-performing classes with improvement suggestions.

## ✅ Solutions <a id='12'></a>

Complete solutions for all exercises are provided below. Study them carefully and compare with your implementations!

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ✅ SOLUTION 1: Multi-Scale Evaluation
# ═══════════════════════════════════════════════════════════════

def exercise_1_solution():
    """Evaluate model performance at multiple input resolutions"""
    
    resolutions = [(64, 64), (128, 128), (256, 256)]
    results = []
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    for idx, (h, w) in enumerate(resolutions):
        # Resize test data
        resized_images = np.array([cv2.resize(img, (w, h)) for img in test_images[:20]])
        resized_masks = np.array([cv2.resize(mask.astype(np.float32), (w, h), 
                                             interpolation=cv2.INTER_NEAREST).astype(np.uint8) 
                                  for mask in test_masks[:20]])
        
        # Simulate predictions (different noise levels for different resolutions)
        noise_level = 0.08 + idx * 0.03  # Higher res = slightly more noise in this simulation
        pred_masks = simulate_predictions(resized_images, resized_masks, noise_level=noise_level)
        
        # Calculate metrics
        metrics = calculate_segmentation_metrics(resized_masks, pred_masks)
        mean_iou = metrics['mean']['IoU']
        
        # Simulate inference time (larger = slower)
        inference_time = 0.005 * (h * w / (64 * 64))  # Simulated ms per image
        
        results.append({
            'resolution': f'{h}×{w}',
            'iou': mean_iou,
            'time_ms': inference_time,
            'pixels': h * w
        })
        
        # Visualize sample prediction
        sample_idx = 5
        mask_colors = np.array([[0.2, 0.2, 0.8], [0.2, 0.8, 0.2], [0.9, 0.9, 0.2]])
        
        axes[0, idx].imshow(resized_images[sample_idx])
        axes[0, idx].set_title(f'{h}×{w}\nInput', fontsize=11, fontweight='bold')
        axes[0, idx].axis('off')
        
        axes[1, idx].imshow(mask_colors[pred_masks[sample_idx]])
        axes[1, idx].set_title(f'Pred (mIoU: {mean_iou:.3f})\nTime: {inference_time:.2f}ms', 
                              fontsize=11, fontweight='bold')
        axes[1, idx].axis('off')
    
    plt.suptitle('✅ Solution 1: Multi-Scale Resolution Analysis', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Trade-off curve
    fig, ax1 = plt.subplots(figsize=(10, 6))
    
    resolutions_labels = [r['resolution'] for r in results]
    ious = [r['iou'] for r in results]
    times = [r['time_ms'] for r in results]
    
    color = 'tab:blue'
    ax1.set_xlabel('Resolution', fontsize=12)
    ax1.set_ylabel('Mean IoU', color=color, fontsize=12)
    ax1.plot(resolutions_labels, ious, color=color, marker='o', linewidth=3, markersize=10, label='IoU')
    ax1.tick_params(axis='y', labelcolor=color)
    ax1.set_ylim(0.5, 1.0)
    ax1.grid(True, alpha=0.3)
    
    ax2 = ax1.twinx()
    color = 'tab:red'
    ax2.set_ylabel('Inference Time (ms)', color=color, fontsize=12)
    ax2.plot(resolutions_labels, times, color=color, marker='s', linewidth=3, markersize=10, label='Time')
    ax2.tick_params(axis='y', labelcolor=color)
    
    plt.title('📈 Accuracy vs. Speed Trade-off', fontsize=14, fontweight='bold')
    fig.tight_layout()
    plt.show()
    
    print("\n📊 Multi-Scale Results Summary:")
    print("=" * 50)
    for r in results:
        print(f"   {r['resolution']:>8s}: IoU={r['iou']:.3f}, Time={r['time_ms']:.2f}ms, Pixels={r['pixels']:,}")
    print("=" * 50)
    print("💡 128×128 offers the best balance for this dataset!")

exercise_1_solution()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ✅ SOLUTION 2: Custom Color Map Generator
# ═══════════════════════════════════════════════════════════════

def exercise_2_solution():
    """Generate rich visualizations with custom colors, patterns, and legends"""
    
    def create_pattern_mask(shape, pattern_type='solid'):
        """Create pattern overlay for segmentation classes"""
        h, w = shape[:2]
        pattern = np.ones((h, w, 3))
        
        if pattern_type == 'stripes':
            for i in range(0, w, 8):
                pattern[:, i:i+4, :] *= 0.7
        elif pattern_type == 'dots':
            for y in range(0, h, 10):
                for x in range(0, w, 10):
                    cv2.circle(pattern, (x, y), 2, (0.7, 0.7, 0.7), -1)
        elif pattern_type == 'checkerboard':
            for y in range(0, h, 10):
                for x in range(0, w, 10):
                    if (x // 10 + y // 10) % 2 == 0:
                        pattern[y:y+10, x:x+10, :] *= 0.75
        
        return pattern
    
    def visualize_with_custom_colormap(image, mask, class_names, class_colors, class_patterns):
        """Create rich visualization with patterns and legend"""
        
        fig = plt.figure(figsize=(16, 10))
        gs = fig.add_gridspec(2, 3, height_ratios=[3, 1], hspace=0.3, wspace=0.3)
        
        # Original
        ax1 = fig.add_subplot(gs[0, 0])
        ax1.imshow(image)
        ax1.set_title('📷 Original Image', fontsize=12, fontweight='bold')
        ax1.axis('off')
        
        # Standard colored mask
        ax2 = fig.add_subplot(gs[0, 1])
        colored_mask = class_colors[mask]
        ax2.imshow(colored_mask)
        ax2.set_title('🎨 Standard Colormap', fontsize=12, fontweight='bold')
        ax2.axis('off')
        
        # Patterned mask
        ax3 = fig.add_subplot(gs[0, 2])
        patterned = np.zeros_like(image)
        
        for cls_idx in range(len(class_names)):
            cls_mask = (mask == cls_idx)
            pattern = create_pattern_mask(image.shape, class_patterns[cls_idx])
            colored_pattern = pattern * class_colors[cls_idx]
            patterned[cls_mask] = colored_pattern[cls_mask]
        
        ax3.imshow(patterned)
        ax3.set_title('✨ Patterned Colormap', fontsize=12, fontweight='bold')
        ax3.axis('off')
        
        # Legend with statistics
        ax4 = fig.add_subplot(gs[1, :])
        ax4.axis('off')
        
        legend_elements = []
        stats_text = "📊 Class Statistics:\n\n"
        
        for i, (name, color, pattern) in enumerate(zip(class_names, class_colors, class_patterns)):
            pixel_count = np.sum(mask == i)
            percentage = pixel_count / mask.size * 100
            
            # Create legend patch
            patch = plt.Rectangle((0, 0), 1, 1, facecolor=color, edgecolor='black', linewidth=2)
            legend_elements.append((patch, f'{name}: {pixel_count:,} px ({percentage:.1f}%)'))
            
            stats_text += f"   {name:12s}: {pixel_count:>6,} px ({percentage:>5.1f}%)  Pattern: {pattern}\n"
        
        # Draw custom legend
        y_pos = 0.8
        for i, (patch, label) in enumerate(legend_elements):
            color = class_colors[i]
            rect = plt.Rectangle((0.05 + i*0.3, y_pos-0.1), 0.05, 0.15, 
                                facecolor=color, edgecolor='black', linewidth=2)
            ax4.add_patch(rect)
            ax4.text(0.11 + i*0.3, y_pos, label, fontsize=11, fontweight='bold', va='center')
        
        ax4.text(0.05, 0.3, stats_text, fontsize=10, family='monospace',
                verticalalignment='top', bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
        
        plt.suptitle('✅ Solution 2: Custom Color Map & Pattern Visualization', 
                    fontsize=15, fontweight='bold')
        plt.show()
    
    # Test on sample
    sample_img = test_images[3]
    sample_mask = test_preds[3]
    
    class_names = ['Background', 'Foreground', 'Border']
    class_colors = np.array([[0.25, 0.25, 0.9], [0.25, 0.9, 0.25], [0.95, 0.85, 0.15]])
    class_patterns = ['checkerboard', 'solid', 'stripes']
    
    visualize_with_custom_colormap(sample_img, sample_mask, class_names, class_colors, class_patterns)

exercise_2_solution()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ✅ SOLUTION 3: Post-Processing Pipeline
# ═══════════════════════════════════════════════════════════════

def exercise_3_solution():
    """Apply morphological post-processing to improve masks"""
    
    def post_process_mask(mask, kernel_size=3):
        """Apply morphological operations to clean mask"""
        processed = mask.copy()
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
        
        # Process each class separately
        result = np.zeros_like(mask)
        
        for cls in range(3):
            cls_mask = (processed == cls).astype(np.uint8)
            
            # Opening: remove small noise
            cls_mask = cv2.morphologyEx(cls_mask, cv2.MORPH_OPEN, kernel)
            
            # Closing: fill small holes
            cls_mask = cv2.morphologyEx(cls_mask, cv2.MORPH_CLOSE, kernel)
            
            result[cls_mask > 0] = cls
        
        return result
    
    def fill_holes(mask):
        """Fill holes in foreground regions"""
        filled = mask.copy()
        
        for cls in [1, 2]:  # Fill holes in foreground and border
            cls_mask = (filled == cls).astype(np.uint8)
            
            # Find contours and fill
            contours, _ = cv2.findContours(cls_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            
            for contour in contours:
                cv2.drawContours(cls_mask, [contour], 0, 1, -1)
            
            filled[cls_mask > 0] = cls
        
        return filled
    
    # Apply post-processing to test predictions
    processed_preds = []
    for pred in test_preds:
        pp = post_process_mask(pred, kernel_size=3)
        pp = fill_holes(pp)
        processed_preds.append(pp)
    
    processed_preds = np.array(processed_preds)
    
    # Compare metrics
    before_metrics = calculate_segmentation_metrics(test_masks, test_preds)
    after_metrics = calculate_segmentation_metrics(test_masks, processed_preds)
    
    # Visualize comparison
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    
    sample_indices = [0, 3, 7]
    mask_colors = np.array([[0.2, 0.2, 0.8], [0.2, 0.8, 0.2], [0.9, 0.9, 0.2]])
    
    for row, idx in enumerate(sample_indices):
        # Original image
        axes[row, 0].imshow(test_images[idx])
        axes[row, 0].set_title(f'Sample {idx}\nOriginal', fontsize=10, fontweight='bold')
        axes[row, 0].axis('off')
        
        # Ground truth
        axes[row, 1].imshow(mask_colors[test_masks[idx]])
        axes[row, 1].set_title('Ground Truth', fontsize=10, fontweight='bold')
        axes[row, 1].axis('off')
        
        # Before post-processing
        axes[row, 2].imshow(mask_colors[test_preds[idx]])
        before_iou = calculate_segmentation_metrics(
            test_masks[idx:idx+1], test_preds[idx:idx+1]
        )['mean']['IoU']
        axes[row, 2].set_title(f'Before Post-Proc\nmIoU: {before_iou:.3f}', 
                              fontsize=10, fontweight='bold')
        axes[row, 2].axis('off')
        
        # After post-processing
        axes[row, 3].imshow(mask_colors[processed_preds[idx]])
        after_iou = calculate_segmentation_metrics(
            test_masks[idx:idx+1], processed_preds[idx:idx+1]
        )['mean']['IoU']
        axes[row, 3].set_title(f'After Post-Proc\nmIoU: {after_iou:.3f}', 
                              fontsize=10, fontweight='bold')
        axes[row, 3].axis('off')
    
    plt.suptitle('✅ Solution 3: Post-Processing Comparison', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Metrics comparison bar chart
    fig, ax = plt.subplots(figsize=(10, 6))
    
    metrics = ['IoU', 'Dice', 'Precision', 'Recall']
    before_vals = [before_metrics['mean'][m] for m in metrics]
    after_vals = [after_metrics['mean'][m] for m in metrics]
    
    x = np.arange(len(metrics))
    width = 0.35
    
    bars1 = ax.bar(x - width/2, before_vals, width, label='Before Post-Processing', 
                   color='lightcoral', edgecolor='black', linewidth=1.5)
    bars2 = ax.bar(x + width/2, after_vals, width, label='After Post-Processing',
                   color='lightgreen', edgecolor='black', linewidth=1.5)
    
    ax.set_ylabel('Score', fontsize=12)
    ax.set_title('📊 Post-Processing Impact on Metrics', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics)
    ax.legend(fontsize=11)
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add value labels
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                   f'{height:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print("\n📈 Post-Processing Results:")
    print("=" * 50)
    for metric in metrics:
        before = before_metrics['mean'][metric]
        after = after_metrics['mean'][metric]
        delta = after - before
        print(f"   {metric:12s}: {before:.3f} → {after:.3f} ({delta:+.3f})")
    print("=" * 50)
    print("💡 Post-processing improves boundary smoothness and removes noise!")

exercise_3_solution()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# ✅ SOLUTION 4: Class-Wise Performance Dashboard
# ═══════════════════════════════════════════════════════════════

def exercise_4_solution():
    """Build comprehensive class-wise performance dashboard"""
    
    from math import pi
    
    # Calculate per-class metrics
    metrics = calculate_segmentation_metrics(test_masks, test_preds)
    
    class_names = ['Background', 'Foreground', 'Border']
    colors = ['#4169E1', '#32CD32', '#FFD700']
    metric_names = ['IoU', 'Dice', 'Precision', 'Recall']
    
    fig = plt.figure(figsize=(18, 12))
    gs = fig.add_gridspec(2, 3, hspace=0.3, wspace=0.3)
    
    # 1. Radar Chart for per-class metrics
    ax1 = fig.add_subplot(gs[0, 0], projection='polar')
    
    categories = metric_names
    N = len(categories)
    angles = [n / float(N) * 2 * pi for n in range(N)]
    angles += angles[:1]
    
    for cls_idx, (cls_name, color) in enumerate(zip(class_names, colors)):
        values = [metrics[cls_idx][m] for m in metric_names]
        values += values[:1]
        
        ax1.plot(angles, values, 'o-', linewidth=2, label=cls_name, color=color)
        ax1.fill(angles, values, alpha=0.15, color=color)
    
    ax1.set_xticks(angles[:-1])
    ax1.set_xticklabels(categories, fontsize=10)
    ax1.set_ylim(0, 1)
    ax1.set_title('🕸️ Per-Class Performance Radar', fontsize=12, fontweight='bold', pad=20)
    ax1.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    ax1.grid(True)
    
    # 2. Grouped bar chart
    ax2 = fig.add_subplot(gs[0, 1:])
    
    x = np.arange(len(metric_names))
    width = 0.25
    
    for cls_idx, (cls_name, color) in enumerate(zip(class_names, colors)):
        values = [metrics[cls_idx][m] for m in metric_names]
        offset = width * (cls_idx - 1)
        bars = ax2.bar(x + offset, values, width, label=cls_name, color=color, 
                       edgecolor='black', linewidth=1.5, alpha=0.85)
        
        for bar, val in zip(bars, values):
            ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                    f'{val:.2f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
    
    ax2.set_ylabel('Score', fontsize=12)
    ax2.set_title('📊 Per-Class Metrics Comparison', fontsize=13, fontweight='bold')
    ax2.set_xticks(x)
    ax2.set_xticklabels(metric_names)
    ax2.legend(fontsize=11)
    ax2.set_ylim(0, 1.1)
    ax2.grid(True, alpha=0.3, axis='y')
    
    # 3. Pixel-level confusion matrix
    ax3 = fig.add_subplot(gs[1, 0])
    
    y_true_flat = test_masks.flatten()
    y_pred_flat = test_preds.flatten()
    
    cm = np.zeros((3, 3), dtype=np.int64)
    for t, p in zip(y_true_flat, y_pred_flat):
        cm[t, p] += 1
    
    # Normalize by row
    cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
    
    im = ax3.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    ax3.set_xticks(range(3))
    ax3.set_yticks(range(3))
    ax3.set_xticklabels(class_names, rotation=45, ha='right')
    ax3.set_yticklabels(class_names)
    ax3.set_xlabel('Predicted', fontsize=11)
    ax3.set_ylabel('True', fontsize=11)
    ax3.set_title('📋 Pixel Confusion Matrix', fontsize=12, fontweight='bold')
    
    for i in range(3):
        for j in range(3):
            text = ax3.text(j, i, f'{cm_norm[i, j]:.2f}\n({cm[i, j]:,})',
                           ha="center", va="center", color="black" if cm_norm[i, j] < 0.5 else "white",
                           fontsize=9, fontweight='bold')
    
    plt.colorbar(im, ax=ax3, fraction=0.046)
    
    # 4. Improvement suggestions panel
    ax4 = fig.add_subplot(gs[1, 1:])
    ax4.axis('off')
    
    # Identify worst performing class
    class_ious = [metrics[i]['IoU'] for i in range(3)]
    worst_idx = np.argmin(class_ious)
    best_idx = np.argmax(class_ious)
    
    suggestions = f"""
📊 Performance Dashboard Analysis
═══════════════════════════════════════════════════════════════

🏆 Best Performing Class: {class_names[best_idx]} (IoU: {class_ious[best_idx]:.3f})
⚠️  Needs Improvement:    {class_names[worst_idx]} (IoU: {class_ious[worst_idx]:.3f})

📈 Improvement Suggestions:

For {class_names[worst_idx]}:
   • Increase weight in loss function (current: {class_weights[worst_idx]:.2f})
   • Add targeted data augmentation (boundary-focused crops)
   • Use larger kernel sizes in decoder for better context
   • Consider deep supervision at multiple scales
   • Apply post-processing (CRF, morphological operations)

General Recommendations:
   • Overall mIoU: {metrics['mean']['IoU']:.3f}
   • {'Excellent' if metrics['mean']['IoU'] > 0.8 else 'Good' if metrics['mean']['IoU'] > 0.7 else 'Needs work'} performance
   • Focus on boundary classes with additional skip connections
   • Try ensemble methods (multiple U-Net variants)
   • Consider larger input resolution for finer details
"""
    
    ax4.text(0.05, 0.5, suggestions, fontsize=10.5, family='monospace',
            verticalalignment='center', bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))
    ax4.set_title('💡 Analysis & Recommendations', fontsize=13, fontweight='bold')
    
    plt.suptitle('✅ Solution 4: Class-Wise Performance Dashboard', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print("\n✅ Dashboard complete! Use these insights to guide model improvements.")

exercise_4_solution()

## 🎓 Summary & Day 89 Teaser <a id='13'></a>

### 🏆 What We Mastered Today:
✅ **Semantic Segmentation fundamentals** — pixel-level classification vs. object detection  
✅ **U-Net architecture** — encoder-decoder with skip connections for precise localization  
✅ **Dataset preparation** — loading, exploring, and analyzing segmentation masks  
✅ **Synchronized augmentation** — spatially consistent transforms for image-mask pairs  
✅ **Model construction** — building U-Net from scratch with Keras Functional API  
✅ **Training strategies** — weighted loss, custom metrics (IoU, Dice), and learning curves  
✅ **Evaluation metrics** — per-class IoU, Dice coefficient, precision, and recall  
✅ **Visualization techniques** — side-by-side comparisons, error heatmaps, and overlays  
✅ **Transfer learning** — leveraging pre-trained backbones (MobileNetV2) for faster convergence  
✅ **Production deployment** — inference pipelines, post-processing, and multi-platform strategies  

### 🚀 Key Takeaways:
- **Skip connections** are the secret sauce of U-Net — they preserve spatial precision lost during downsampling
- **Class imbalance** is ubiquitous in segmentation; use weighted loss or focal loss to handle it
- **IoU and Dice** are more meaningful than pixel accuracy for evaluating segmentation quality
- **Post-processing** (morphological operations, CRF) can significantly improve boundary quality
- **Transfer learning** from ImageNet features accelerates training and improves generalization

---

### 🔮 Day 89 Teaser:
Tomorrow we explore **Generative Adversarial Networks (GANs)**! We'll learn how to build and train GANs to generate realistic synthetic images, understand the adversarial training dynamics between generator and discriminator, and create our own image generation pipeline. Get ready to teach machines to be creative! 🎨✨

---

> *"Every pixel tells a story — segmentation gives us the vocabulary to read it."* — Keep segmenting, keep learning! 🚀

**⭐ Day 88 Complete! See you tomorrow for Day 89! ⭐**